In [1]:
!uname -a && cat /etc/*release
!pwd
!ls -la

Linux 46e572a5d438 6.6.113+ #1 SMP Mon Feb  2 12:27:57 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux
DISTRIB_ID=Ubuntu
DISTRIB_RELEASE=22.04
DISTRIB_CODENAME=jammy
DISTRIB_DESCRIPTION="Ubuntu 22.04.5 LTS"
PRETTY_NAME="Ubuntu 22.04.5 LTS"
NAME="Ubuntu"
VERSION_ID="22.04"
VERSION="22.04.5 LTS (Jammy Jellyfish)"
VERSION_CODENAME=jammy
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=jammy
/content
total 16
drwxr-xr-x 1 root root 4096 Feb  6 14:31 .
drwxr-xr-x 1 root root 4096 Mar 23 15:57 ..
drwxr-xr-x 4 root root 4096 Feb  6 14:31 .config
drwxr-xr-x 1 root root 4096 Feb  6 14:31 sample_data


In [15]:
cell_str='''
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>
#include <cuda_runtime.h>

// --- KERNEL NAIVE ---
// Nessuna ottimizzazione, nessuna memoria condivisa.
// Ogni thread legge direttamente dalla lentissima Memoria Globale.
__global__ void matMulNaive(const float *a, const float *b, float *c, int n) {
    // I cicli "i" e "j" scompaiono: le coordinate sono calcolate tramite gli indici dei thread
    int row = blockIdx.y * blockDim.y + threadIdx.y; // Sostituisce la 'i'
    int col = blockIdx.x * blockDim.x + threadIdx.x; // Sostituisce la 'j'

    // Controllo dei bordi: ci assicuriamo che il thread sia dentro la matrice
    if (row < n && col < n) {
        float sum = 0.0;

        // Questo è l'unico ciclo che sopravvive (corrisponde al ciclo 'k' originale)
        for (int k = 0; k < n; ++k) {
            // Lettura pesantissima direttamente dalla memoria globale (VRAM)
            sum += a[row * n + k] * b[k * n + col];
        }

        c[row * n + col] = sum;
    }
}

int main(int argc, char **argv) {
    if (argc < 2) {
        fprintf(stderr, "Error: missing n argument.\\n");
        fprintf(stderr, "Usage: %s <n>\\n", argv[0]);
        return 1;
    }

    int n = atoi(argv[1]);
    if (n <= 0) {
        fprintf(stderr, "Error: You forgot to provide n!\\n");
        return 1;
    }

    // Calcoliamo i byte necessari (usando double: 8 byte per elemento)
    size_t bytes = (size_t)n * n * sizeof(float);

    // Allocazione Host (CPU)
    // NOTA: In CUDA si preferisce sempre "appiattire" le matrici in array 1D
    float *h_a = (float*)malloc(bytes);
    float *h_b = (float*)malloc(bytes);
    float *h_c = (float*)malloc(bytes);

    if (!h_a || !h_b || !h_c) {
        fprintf(stderr, "Errore di allocazione RAM Host!\\n");
        return 1;
    }

    // Inizializzazione
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            h_a[i * n + j] = 2.0;
            h_b[i * n + j] = 3.0;
            h_c[i * n + j] = 0.0;
        }
    }

    // Allocazione Device (GPU)
    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    // Trasferimento dati HtoD
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // Configurazione della Griglia di esecuzione (Blocchi 32x32 = 1024 thread)
    int blockSize = 32;
    dim3 threadsPerBlock(blockSize, blockSize);
    dim3 numBlocks((n + blockSize - 1) / blockSize, (n + blockSize - 1) / blockSize);

    printf("Starting the computation (CUDA Naive - FP64)...\\n");
    clock_t start = clock();

    // Lancio del kernel
    matMulNaive<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, n);

    // La CPU deve aspettare che la GPU finisca prima di fermare il cronometro
    cudaDeviceSynchronize();
    clock_t end = clock();

    double duration = (double)(end - start) / CLOCKS_PER_SEC;
    printf("Execution Time: %.4f seconds\\n", duration);

    // Trasferimento risultati DtoH
    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

    // Salvataggio dei risultati (con salvaguardia per n piccoli)
    FILE *f = fopen("mat-res.txt", "w");
    if (!f) {
        perror("fopen");
        return 1;
    }

    fprintf(f, "%d\\n\\n", n);
    int sample_size = (n < 1000) ? n : 1000;
    for (int i = 0; i < sample_size; i++) {
        for (int j = 0; j < sample_size; j++) {
            fprintf(f, "%.0f ", h_c[i * n + j]);
        }
        fprintf(f, "\\n");
    }

    fclose(f);

    // Pulizia finale
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(h_a);
    free(h_b);
    free(h_c);

    return 0;
}
'''

with open('cuda_matrixmult_naive.cu', 'w') as f:
    f.write(cell_str)

In [16]:
!nvcc -arch=sm_75 -O3 cuda_matrixmult_naive.cu -o cuda_matrixmult_naive
!nvprof ./cuda_matrixmult_naive 20000

==6013== NVPROF is profiling process 6013, command: ./cuda_matrixmult_naive 20000
Starting the computation (CUDA Naive - FP64)...
Execution Time: 31.3668 seconds
==6013== Profiling application: ./cuda_matrixmult_naive 20000
==6013== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   96.79%  31.5601s         1  31.5601s  31.5601s  31.5601s  matMulNaive(float const *, float const *, float*, int)
                    2.07%  676.20ms         2  338.10ms  335.87ms  340.32ms  [CUDA memcpy HtoD]
                    1.14%  371.79ms         1  371.79ms  371.79ms  371.79ms  [CUDA memcpy DtoH]
      API calls:   96.21%  31.5602s         1  31.5602s  31.5602s  31.5602s  cudaDeviceSynchronize
                    3.20%  1.04884s         3  349.61ms  336.09ms  372.22ms  cudaMemcpy
                    0.56%  185.08ms         3  61.695ms  155.34us  184.77ms  cudaMalloc
                    0.02%  6.2408ms         3  2.0803ms  1.3694ms  2